# Multilayer target and `run.yaml`

A layered target, described once as a **run spec** (`run.yaml`, see the
[run.yaml page](https://pysrim2.readthedocs.io/en/latest/run-spec.html)): the spec holds everything needed to regenerate
`TRIM.IN`, with the unit in each key name (`energy_keV`, `thickness_nm`, ...), and the same
file drives `TRIM.from_spec`, `pysrim2 run` and the exports. Set up SRIM once with `pysrim2 install` and check it with `pysrim2 doctor`
(see the [installation guide](https://pysrim2.readthedocs.io/en/latest/installation.html)). The notebook then finds SRIM
by itself, natively or in the container; no path is needed. SRIM-2013 only, **not** 2018.
SRIM is (c) SRIM.com (James F. Ziegler), free for non-commercial use only; see `LEGAL.rtf` next to your SRIM copy.

In [ ]:
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt

from srim import TRIM, Ion, Layer, Target
from srim.spec import RunSpec

OUT = Path("multilayer-output")
OUT.mkdir(exist_ok=True)

## Build the target in code

In [ ]:
ion = Ion("Ni", energy=3.0e6)

ni = Layer({"Ni": {"stoich": 1.0, "E_d": 30.0, "lattice": 0.0, "surface": 3.0}},
           density=8.9, width=20000.0, name="Ni")
sic_ni = Layer(
    {
        "Si": {"stoich": 1 / 3, "E_d": 25.0, "lattice": 0.0, "surface": 3.0},
        "C": {"stoich": 1 / 3, "E_d": 25.0, "lattice": 0.0, "surface": 3.0},
        "Ni": {"stoich": 1 / 3, "E_d": 25.0, "lattice": 0.0, "surface": 3.0},
    },
    density=3.21,
    width=15000.0,
    name="SiCNi",
)
trim = TRIM(Target([ni, sic_ni]), ion, number_ions=20, calculation=1, random_seed=12345)

## Describe it as a run spec

`to_yaml` records the seed (one is picked if you give none), so the file reproduces the run.

In [ ]:
trim.to_yaml(OUT / "run.yaml")
print((OUT / "run.yaml").read_text())

In [ ]:
spec = RunSpec.from_yaml(OUT / "run.yaml")
spec.ion, [layer.name for layer in spec.target]

## Run it from the spec

`TRIM.from_spec(spec)` writes a byte-identical `TRIM.IN` to the object above.

In [ ]:
trim2 = TRIM.from_spec(spec)
results = trim2.run()  # also writes run.yaml, with provenance, next to SRIM's outputs
results.vacancy.df.head()

In [ ]:
vac = results.vacancy.df
total = vac["knock_ons_per_A_per_ion"] + vac.filter(like="vacancies_").sum(axis=1)

fig, ax = plt.subplots()
ax.plot(vac["depth_A"], total)
for boundary in (20000.0,):  # the layer interface [A]
    ax.axvline(boundary, color="gray", ls=":")
ax.set(xlabel="Depth [A]", ylabel="vacancies / (A ion)", title="Vacancies in the two layers")
plt.show()

## The same from the command line

`pysrim2 run` takes the spec, overrides values with `--set dotted.key=value`, runs TRIM and
copies the outputs and a `run.yaml` with provenance into `--output`.

In [ ]:
cmd = [sys.executable, "-m", "srim", "run", str(OUT / "run.yaml"),
       "--set", "ion.energy_keV=2000", "--set", "number_ions=10",
       "--output", str(OUT / "cli-run")]
print(subprocess.run(cmd, capture_output=True, text=True, check=True).stdout)
sorted(p.name for p in (OUT / "cli-run").iterdir())

In [ ]:
RunSpec.from_yaml(OUT / "cli-run" / "run.yaml").ion  # the spec that was actually run